# AI Model Router — Complexity Classifier

Rule-based classifier that scores a question 0–100 and routes it to the cheapest
Gemini model that can handle it:

| Score | Tier | Model |
|------|------|-------|
| 0–14 | light | `gemini-3.5-flash-lite` |
| 15–55 | mid | `gemini-3.6-flash` |
| 56–100 | heavy | `gemini-3.1-pro` |

**Run top to bottom.** Cell 1 = the classifier, Cell 2 = quick demo,
Cell 3 = accuracy + confusion matrix, Cell 4 = the tuning loop.
This notebook is self-contained (dataset is inline) — no other files needed.

## 1. The classifier

In [ ]:
"""
Rule-based question complexity classifier (Python port of classifier.js).
Scores a prompt 0-100, then routes to one of three Gemini tiers.
Pure heuristics, no ML, no network calls.

    from classifier import classify, route_model
    r = classify("Why does quicksort degrade to O(n^2)?")
    r["model"]     # Gemini model id to call
    r["score"]     # 0-100 complexity
    r["tier"]      # "light" | "mid" | "heavy"
    r["features"]  # per-signal breakdown for your analytics page
"""
import re

# ------------------------------------------------------------------ #
# 1. Model tiers  (edit `model` strings to your real API ids)
# ------------------------------------------------------------------ #
TIERS = {
    "LIGHT": {"name": "light", "model": "gemini-3.5-flash-lite",
              "label": "Flash Lite", "max_score": 14},
    "MID":   {"name": "mid",   "model": "gemini-3.6-flash",
              "label": "Flash",      "max_score": 55},
    "HEAVY": {"name": "heavy", "model": "gemini-3.1-pro",
              "label": "Pro",        "max_score": 100},
}

# ------------------------------------------------------------------ #
# 2. Signal dictionaries
# ------------------------------------------------------------------ #
REASONING_WORDS = [
    "why", "how", "explain", "analyze", "analyse", "compare", "contrast",
    "evaluate", "assess", "justify", "prove", "derive", "design", "architect",
    "optimize", "optimise", "refactor", "debug", "troubleshoot", "diagnose",
    "trade-off", "tradeoff", "implications", "strategy", "reason", "cause",
    "critique", "synthesize", "synthesise", "implement", "algorithm",
]

SIMPLE_WORDS = [
    "define", "definition", "what is", "what's", "who is", "who's", "when is",
    "where is", "capital of", "translate", "spell", "meaning of", "convert",
    "how many", "how much", "list", "name a", "abbreviation", "synonym",
    "antonym", "yes or no",
]

HARD_DOMAINS = [
    "legal", "lawsuit", "contract", "regulation", "medical", "diagnosis",
    "clinical", "quantum", "cryptograph", "differential equation", "tensor",
    "distributed system", "concurrency", "kubernetes", "compiler",
    "machine learning", "neural network", "financial model", "tax", "actuarial",
    "microservice", "event-driven", "kafka", "rabbitmq", "consensus", "raft",
    "multi-tenant", "database schema", "row-level", "backpropagation",
    "undecidable", "halting problem", "race condition", "deadlock",
    "fault-tolerant", "theorem", "diagonalization",
]

TASK_WORDS = [
    "write", "rewrite", "draft", "compose", "generate", "create", "summarize",
    "summarise", "plan", "outline", "rephrase", "paraphrase", "brainstorm",
    "make a", "build a", "give me a",
]

CODE_KEYWORDS = [
    "function", "class ", "def ", "async", "await", "import", "const ", "let ",
    "var ", "return", "for (", "while (", "=>", "null", "undefined", "python",
    "javascript", "typescript", "java", "c++", "rust", "sql", "regex", "api",
    "stack trace", "exception", "error:", "npm", "git ",
]

# ------------------------------------------------------------------ #
# 3. Scoring weights  (tune these — they're the whole model)
# ------------------------------------------------------------------ #
W = {
    "length_per_word": 0.7, "length_cap": 28,
    "reasoning_word": 11,   "reasoning_cap": 33,
    "question": 7,          "question_cap": 21,
    "code": 26,
    "math": 14,
    "hard_domain": 27,
    "multi_step": 12,
    "task_word": 13,
    "constraint": 5,        "constraint_cap": 15,
    "combo": 9,
    "simple_word": -14,     "simple_cap": -28,
    "short_bonus": -12,
}

# ------------------------------------------------------------------ #
# 4. Helpers
# ------------------------------------------------------------------ #
def _count(text, needles):
    return sum(1 for w in needles if w in text)

def _clamp(v, lo, hi):
    return max(lo, min(hi, v))

def _round_half_up(v):
    # Match JavaScript's Math.round (round half up), not Python's round-to-even.
    import math
    return math.floor(v + 0.5)

# ------------------------------------------------------------------ #
# 5. Core classifier
# ------------------------------------------------------------------ #
def classify(prompt=""):
    raw = str(prompt)
    text = raw.lower()
    words = [w for w in text.split() if w]
    word_count = len(words)

    features = {}
    score = 0.0

    def add(key, value, points):
        nonlocal score
        features[key] = {"value": value, "points": round(points, 1)}
        score += points

    # length
    add("word_count", word_count,
        _clamp(word_count * W["length_per_word"], 0, W["length_cap"]))

    # reasoning keywords
    reasoning_hits = _count(text, REASONING_WORDS)
    add("reasoning_words", reasoning_hits,
        _clamp(reasoning_hits * W["reasoning_word"], 0, W["reasoning_cap"]))

    # multiple questions
    question_marks = raw.count("?")
    add("question_marks", question_marks,
        _clamp(max(0, question_marks - 1) * W["question"], 0, W["question_cap"]))

    # code
    has_code = "```" in raw or _count(text, CODE_KEYWORDS) > 0
    add("code", has_code, W["code"] if has_code else 0)

    # math
    has_math = bool(
        re.search(r"[=+\-*/^]\s*\d", raw)
        or re.search(r"\b(integral|derivative|equation|matrix|probability|"
                     r"theorem|solve for|sqrt|log)\b", text)
        or re.search(r"\d+\s*[+\-*/^]\s*\d+", raw)
    )
    add("math", has_math, W["math"] if has_math else 0)

    # specialist domain
    has_hard_domain = _count(text, HARD_DOMAINS) > 0
    add("hard_domain", has_hard_domain, W["hard_domain"] if has_hard_domain else 0)

    # multi-step
    multi_step = bool(re.search(
        r"step[- ]by[- ]step|and then|first.*then|\b1\.\s.*\b2\.\s|"
        r"walk me through|\d+[- ]step", text))
    add("multi_step", multi_step, W["multi_step"] if multi_step else 0)

    # generation / instruction verb
    has_task_word = _count(text, TASK_WORDS) > 0
    add("task_word", has_task_word, W["task_word"] if has_task_word else 0)

    # constraints (conjunctions)
    constraints = len(re.findall(
        r"\b(and|also|plus|as well as|in addition|but|however|except|without)\b",
        text))
    add("constraints", constraints,
        _clamp(constraints * W["constraint"], 0, W["constraint_cap"]))

    # simple lookup phrases (negative)
    simple_hits = _count(text, SIMPLE_WORDS)
    add("simple_words", simple_hits,
        _clamp(simple_hits * W["simple_word"], W["simple_cap"], 0))

    # very short prompt (negative)
    is_short = 0 < word_count <= 4
    add("short_prompt", is_short, W["short_bonus"] if is_short else 0)

    # combo bonus: stacked hard signals reinforce each other
    combo = (has_hard_domain and reasoning_hits > 0) or (has_code and reasoning_hits > 0)
    add("combo_bonus", combo, W["combo"] if combo else 0)

    score = int(_clamp(_round_half_up(score), 0, 100))
    tier = route_tier(score)
    return {
        "score": score,
        "tier": tier["name"],
        "model": tier["model"],
        "label": tier["label"],
        "features": features,
    }

# ------------------------------------------------------------------ #
# 6. Routing
# ------------------------------------------------------------------ #
def route_tier(score):
    if score <= TIERS["LIGHT"]["max_score"]:
        return TIERS["LIGHT"]
    if score <= TIERS["MID"]["max_score"]:
        return TIERS["MID"]
    return TIERS["HEAVY"]

def route_model(prompt):
    return classify(prompt)["model"]


## 2. Quick demo — try your own prompts

In [ ]:
samples = [
    "capital of France?",
    "define osmosis",
    "How do I center a div in CSS?",
    "Why does quicksort degrade to O(n^2)?",
    "Explain how attention works in transformers and compare it to RNNs.",
    "Design a distributed rate limiter and prove its consistency.",
]
for q in samples:
    r = classify(q)
    print(f"{r['score']:>3}  {r['label']:<11} -> {r['model']:<22} | {q[:55]}")

# peek at WHY a prompt scored what it did (great for your analytics page):
from pprint import pprint
print("\n--- feature breakdown ---")
pprint(classify("Design a distributed rate limiter and prove its consistency.")["features"])

## 3. Accuracy + confusion matrix\n\nThe labeled dataset is inline below. `tier` is the answer you *want*; the classifier's pick is compared against it.

In [ ]:
DATASET = [
  {
    "q": "capital of France?",
    "tier": "light"
  },
  {
    "q": "define osmosis",
    "tier": "light"
  },
  {
    "q": "what's the boiling point of water in Celsius?",
    "tier": "light"
  },
  {
    "q": "translate 'good morning' to Spanish",
    "tier": "light"
  },
  {
    "q": "who is the CEO of Apple?",
    "tier": "light"
  },
  {
    "q": "convert 10 miles to kilometers",
    "tier": "light"
  },
  {
    "q": "how many continents are there?",
    "tier": "light"
  },
  {
    "q": "spell 'necessary'",
    "tier": "light"
  },
  {
    "q": "synonym for happy",
    "tier": "light"
  },
  {
    "q": "what year did WW2 end?",
    "tier": "light"
  },
  {
    "q": "list three primary colors",
    "tier": "light"
  },
  {
    "q": "abbreviation for California",
    "tier": "light"
  },
  {
    "q": "hi",
    "tier": "light"
  },
  {
    "q": "what does HTTP stand for?",
    "tier": "light"
  },
  {
    "q": "How do I center a div in CSS?",
    "tier": "mid"
  },
  {
    "q": "Summarize the plot of Hamlet in two sentences.",
    "tier": "mid"
  },
  {
    "q": "Explain the difference between let and var in JavaScript.",
    "tier": "mid"
  },
  {
    "q": "Why is the sky blue?",
    "tier": "mid"
  },
  {
    "q": "Write a short poem about autumn.",
    "tier": "mid"
  },
  {
    "q": "How does a for loop work in Python?",
    "tier": "mid"
  },
  {
    "q": "Compare cats and dogs as pets.",
    "tier": "mid"
  },
  {
    "q": "Explain how attention works in transformers and compare it to RNNs.",
    "tier": "mid"
  },
  {
    "q": "What are the pros and cons of remote work?",
    "tier": "mid"
  },
  {
    "q": "Solve for x: 3x + 5 = 20 and explain the step.",
    "tier": "mid"
  },
  {
    "q": "Explain what a REST API is with an example.",
    "tier": "mid"
  },
  {
    "q": "Why does quicksort degrade to O(n^2)?",
    "tier": "mid"
  },
  {
    "q": "Give me a 5-step plan to learn guitar.",
    "tier": "mid"
  },
  {
    "q": "Rewrite this sentence to be more formal: hey can u send the doc",
    "tier": "mid"
  },
  {
    "q": "Design a distributed rate limiter for a kubernetes cluster handling 1M req/s and prove its consistency.",
    "tier": "heavy"
  },
  {
    "q": "Debug this: function f(){ for(let i=0;i<n;i++){ return i } } why does it only return once and how do I fix it?",
    "tier": "heavy"
  },
  {
    "q": "Derive the backpropagation equations for a two-layer neural network and explain each term.",
    "tier": "heavy"
  },
  {
    "q": "Design a database schema for a multi-tenant SaaS with row-level security, and justify the tradeoffs.",
    "tier": "heavy"
  },
  {
    "q": "Analyze the legal implications of this contract clause and compare it to standard indemnification terms.",
    "tier": "heavy"
  },
  {
    "q": "Optimize this SQL query that joins five tables and explain why it's slow, then rewrite it.",
    "tier": "heavy"
  },
  {
    "q": "Prove that the halting problem is undecidable using a diagonalization argument.",
    "tier": "heavy"
  },
  {
    "q": "Architect a fault-tolerant event-driven microservice system and evaluate the tradeoffs of Kafka vs RabbitMQ.",
    "tier": "heavy"
  },
  {
    "q": "Solve this differential equation dy/dx = xy with initial condition y(0)=1 and explain each step.",
    "tier": "heavy"
  },
  {
    "q": "Refactor this concurrency code to avoid the race condition and explain why the deadlock happens.",
    "tier": "heavy"
  },
  {
    "q": "Explain how to implement a Raft consensus algorithm and prove its safety guarantees.",
    "tier": "heavy"
  },
  {
    "q": "Diagnose why this distributed system loses messages under network partition and design a fix.",
    "tier": "heavy"
  }
]

ORDER = ["light", "mid", "heavy"]

def evaluate(dataset):
    m = {e: {p: 0 for p in ORDER} for e in ORDER}
    correct, misses = 0, []
    for item in dataset:
        r = classify(item["q"])
        p = r["tier"]
        m[item["tier"]][p] += 1
        if p == item["tier"]:
            correct += 1
        else:
            misses.append((item["tier"], p, r["score"], item["q"]))
    total = len(dataset)
    print(f"=== Accuracy: {correct}/{total} ({correct/total*100:.1f}%) ===\n")
    print("Confusion matrix (rows=expected, cols=predicted):")
    print("            light   mid   heavy")
    for e in ORDER:
        print(f"  {e:<8}" + "".join(f"{m[e][p]:>6}" for p in ORDER))
    print("\nPer-tier recall:")
    for e in ORDER:
        tot = sum(m[e].values())
        rec = (m[e][e] / tot * 100) if tot else 0
        print(f"  {e:<8} {rec:>3.0f}%  ({m[e][e]}/{tot})")
    if misses:
        print(f"\nMisses ({len(misses)}) — tune for these:")
        for want, got, sc, q in misses:
            print(f"  [want {want}, got {got} @{sc}] {q[:65]}")
    return correct / total

evaluate(DATASET);

## 4. The tuning loop

To make it more accurate: change a weight in the `W` dict (Cell 1) or a tier
threshold in `TIERS`, **re-run Cell 1, then re-run Cell 3**, and watch the
confusion matrix. A miss that scored *too low* needs more points (bump a weight)
or a lower threshold; too high, the reverse.

**Biggest win:** grow `DATASET`. Log real prompts from your extension, label each
with the tier you think is right, paste them into the `DATASET` list in Cell 3,
re-run. 90%+ on a *large, realistic* set beats 100% on these 40 examples
(that's just overfitting the keywords).

In [ ]:
# Add your own labeled examples here, then re-run to see the effect:
MY_EXAMPLES = [
    {"q": "what time is it in Tokyo?", "tier": "light"},
    {"q": "explain the CAP theorem with an example", "tier": "mid"},
    {"q": "prove that P != NP is still open and summarize the main approaches", "tier": "heavy"},
]
evaluate(DATASET + MY_EXAMPLES);